In [ ]:
import pandas as pd
import numpy as np
import pickle, os
from matplotlib import pyplot as plt
from tqdm import tqdm


In [ ]:
def load_walker_data_colvar(walker_id, base_path='/Volumes/LaCie/STEPs/ConvergenceTest/walkers'):
    """Load data from a single walker using COLVAR files"""
    # COLVAR files are numbered 0-7, while folders are 1-8
    colvar_file_number = walker_id - 1  # Convert walker_id (1-8) to COLVAR number (0-7)
    colvar_file = f'{base_path}/{walker_id}/COLVAR.{colvar_file_number}'
    
    # Step 1: Read the column names from the line starting with "#! FIELDS"
    with open(colvar_file, 'r') as f:
        for line in f:
            if line.startswith('#! FIELDS'):
                columns = line.strip().split()[2:]  # Skip '#!', 'FIELDS'
                break
    
    # Step 2: Use `pd.read_csv` with `skiprows`
    def skip_every_20th(start, total_lines):
        return [i for i in range(start, total_lines) if i % 20 != 0]
    
    with open(colvar_file, 'r') as f:
        total_rows = sum(1 for line in f if not line.startswith('#'))
    
    df = pd.read_csv(
        colvar_file,
        delim_whitespace=True,
        comment='#',
        header=None,
        names=columns,
        skiprows=skip_every_20th(1, total_rows)
    )
    
    return df

def preprocess_walker_data(df):
    """Apply your preprocessing to a walker's data"""
    # Convert object columns to float
    object_columns = df.select_dtypes(include=['object']).columns
    df[object_columns] = df[object_columns].apply(pd.to_numeric, errors='coerce')
    
    # Convert to degrees
    df['om1'] = df['omega1'] * (180/np.pi)
    df['om2'] = df['omega2'] * (180/np.pi)
    df['om3'] = df['omega3'] * (180/np.pi)
    
    # Apply your conditions
    conditions = [
        ((df['om1'] >=-45) & (df['om1'] <=45)) & ((df['om2'] >=-45) & (df['om2'] <=45)) & ((df['om3'] >=-45) & (df['om3'] <=45)),
        ((df['om1'] >=-45) & (df['om1'] <=45)) & ((df['om2'] >=-45) & (df['om2'] <=45)) & ((df['om3'] <=-135) | (df['om3'] >=135)),
        ((df['om1'] >=-45) & (df['om1'] <=45)) & ((df['om2'] <=-135) | (df['om2'] >=135)) & ((df['om3'] >=-45) & (df['om3'] <=45)),
        ((df['om1'] <=-135) | (df['om1'] >=135)) & ((df['om2'] >=-45) & (df['om2'] <=45)) & ((df['om3'] >=-45) & (df['om3'] <=45)),
        ((df['om1'] <=-135) | (df['om1'] >=135)) & ((df['om2'] <=-135) | (df['om2'] >=135)) & ((df['om3'] >=-45) & (df['om3'] <=45)),
        ((df['om1'] <=-135) | (df['om1'] >=135)) & ((df['om2'] >=-45) & (df['om2'] <=45)) & ((df['om3'] <=-135) | (df['om3'] >=135)),
        ((df['om1'] >=-45) & (df['om1'] <=45)) & ((df['om2'] <=-135) | (df['om2'] >=135)) & ((df['om3'] <=-135) | (df['om3'] >=135)),
        ((df['om1'] <=-135) | (df['om1'] >=135)) & ((df['om2'] <=-135) | (df['om2'] >=135)) & ((df['om3'] <=-135) | (df['om3'] >=135))
    ]
    choices = [1,2,3,4,5,6,7,8]
    df['c'] = np.select(conditions, choices, default=0)
    
    return df

# Load and preprocess all walker data from COLVAR files
print("Loading COLVAR walker data...")
walker_data_colvar = {}
for walker_id in range(1, 9):
    print(f"Loading walker {walker_id} (COLVAR.{walker_id-1})...")
    try:
        df_walker = load_walker_data_colvar(walker_id)
        df_walker = preprocess_walker_data(df_walker)
        walker_data_colvar[walker_id] = df_walker
        print(f"Walker {walker_id}: {len(df_walker)} frames")
    except FileNotFoundError:
        print(f"Warning: COLVAR.{walker_id-1} not found in folder {walker_id}")
    except Exception as e:
        print(f"Error loading walker {walker_id}: {e}")

print(f"Successfully loaded {len(walker_data_colvar)} walkers")

# Option 1: Combine all walkers into one dataset
def combine_all_walkers(walker_data):
    """Combine all walker data into one large dataset"""
    combined_df = pd.concat([walker_data[i] for i in range(1, 9)], ignore_index=True)
    return combined_df

# Option 2: Calculate convergence for each walker separately, then average
def calculate_walker_convergence(walker_data, window_size=1000):
    """Calculate convergence for each walker separately"""
    walker_convergence = {}
    
    for walker_id in range(1, 9):
        print(f"Processing walker {walker_id}...")
        df = walker_data[walker_id]
        
        frames = range(window_size, len(df), window_size)
        convergence_data = {
            'frames': [],
            'populations': {state: [] for state in range(1, 9)},
            'free_energies': {state: [] for state in range(1, 9)}
        }
        
        for frame in frames:
            pops, fes = calculate_cumulative_populations(df, frame)
            convergence_data['frames'].append(frame)
            
            for state in range(1, 9):
                convergence_data['populations'][state].append(pops[state])
                convergence_data['free_energies'][state].append(fes[state])
        
        walker_convergence[walker_id] = convergence_data
    
    return walker_convergence

def calculate_cumulative_populations(df, up_to_frame):
    """Your existing function"""
    subset_df = df.iloc[:up_to_frame]
    global_min = np.exp(subset_df['pb.bias']/2.5).sum()
    
    populations = {}
    free_energies = {}
    
    for state in range(1, 9):
        state_data = subset_df[subset_df['c'] == state]
        if len(state_data) > 0:
            pop = np.exp(state_data['pb.bias']/2.5).sum() / global_min
            populations[state] = pop
            if pop > 0:
                free_energies[state] = -0.6 * np.log(pop)
            else:
                free_energies[state] = np.inf
        else:
            populations[state] = 0
            free_energies[state] = np.inf
    
    return populations, free_energies

# Calculate convergence for each walker
walker_convergence = calculate_walker_convergence(walker_data_colvar, window_size=1000)

# Calculate average convergence across walkers
def average_walker_convergence(walker_convergence):
    """Average convergence data across all walkers"""
    # Find common time points (minimum length across walkers)
    min_length = min(len(walker_convergence[i]['frames']) for i in range(1, 9))
    
    averaged_convergence = {
        'frames': walker_convergence[1]['frames'][:min_length],
        'populations': {state: [] for state in range(1, 9)},
        'free_energies': {state: [] for state in range(1, 9)},
        'pop_std': {state: [] for state in range(1, 9)},
        'fe_std': {state: [] for state in range(1, 9)}
    }
    
    for time_idx in range(min_length):
        for state in range(1, 9):
            # Collect values from all walkers at this time point
            pops = [walker_convergence[walker]['populations'][state][time_idx] 
                   for walker in range(1, 9)]
            fes = [walker_convergence[walker]['free_energies'][state][time_idx] 
                  for walker in range(1, 9)]
            
            # Remove infinite values for averaging
            finite_fes = [fe for fe in fes if fe != np.inf and not np.isnan(fe)]
            
            # Calculate averages and standard deviations
            averaged_convergence['populations'][state].append(np.mean(pops))
            averaged_convergence['pop_std'][state].append(np.std(pops))
            
            if finite_fes:
                averaged_convergence['free_energies'][state].append(np.mean(finite_fes))
                averaged_convergence['fe_std'][state].append(np.std(finite_fes))
            else:
                averaged_convergence['free_energies'][state].append(np.inf)
                averaged_convergence['fe_std'][state].append(0)
    
    return averaged_convergence

# Calculate averaged convergence
print("Averaging across walkers...")
averaged_convergence = average_walker_convergence(walker_convergence)

# Convert to time
dt = 0.002  # adjust according to your simulation
time_ps = [frame * dt for frame in averaged_convergence['frames']]

# Plot averaged convergence with walker-to-walker error bars
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

colors = plt.cm.tab10(np.linspace(0, 1, 8))

# Plot population convergence with walker variability
for i, state in enumerate(range(1, 9)):
    pops = averaged_convergence['populations'][state]
    errors = averaged_convergence['pop_std'][state]
    
    ax1.plot(time_ps, pops, label=f'{state_labels[state]}', color=colors[i], linewidth=2)
    ax1.fill_between(time_ps, 
                     [p - e for p, e in zip(pops, errors)],
                     [p + e for p, e in zip(pops, errors)],
                     color=colors[i], alpha=0.3)

ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Population')
ax1.set_title('Population Convergence (Averaged Across 8 Walkers)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Plot free energy convergence with walker variability
for i, state in enumerate(range(1, 9)):
    fes = [fe if fe != np.inf else np.nan for fe in averaged_convergence['free_energies'][state]]
    errors = averaged_convergence['fe_std'][state]
    
    ax2.plot(time_ps, fes, label=f'{state_labels[state]}', color=colors[i], linewidth=2)
    ax2.fill_between(time_ps,
                     [f - e if not np.isnan(f) else f for f, e in zip(fes, errors)],
                     [f + e if not np.isnan(f) else f for f, e in zip(fes, errors)],
                     color=colors[i], alpha=0.3)

ax2.set_xlabel('Time (ps)')
ax2.set_ylabel('Free Energy (kcal/mol)')
ax2.set_title('Free Energy Convergence (Averaged Across 8 Walkers)')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final averaged results
print("\nFinal averaged results across all walkers:")
for state in range(1, 9):
    final_pop = averaged_convergence['populations'][state][-1]
    final_pop_std = averaged_convergence['pop_std'][state][-1]
    final_fe = averaged_convergence['free_energies'][state][-1]
    final_fe_std = averaged_convergence['fe_std'][state][-1]
    
    print(f"{state_labels[state]}: P = {final_pop:.4f} ± {final_pop_std:.4f}, "
          f"dG = {final_fe:.2f} ± {final_fe_std:.2f} kcal/mol")

# Optional: Plot individual walker trajectories for comparison
def plot_individual_walkers(walker_convergence, state_to_plot=1):
    """Plot individual walker trajectories for a specific state"""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for walker_id in range(1, 9):
        convergence = walker_convergence[walker_id]
        time_ps_walker = [frame * dt for frame in convergence['frames']]
        pops = convergence['populations'][state_to_plot]
        
        ax.plot(time_ps_walker, pops, alpha=0.7, label=f'Walker {walker_id}')
    
    # Plot average
    avg_pops = averaged_convergence['populations'][state_to_plot]
    ax.plot(time_ps, avg_pops, 'k-', linewidth=3, label='Average')
    
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Population')
    ax.set_title(f'Individual Walker Convergence - {state_labels[state_to_plot]}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

# Example: Plot individual trajectories for CCC state
plot_individual_walkers(walker_convergence, state_to_plot=1)

In [ ]:
import pickle
from pathlib import Path

def save_convergence_data(walker_data, convergence_results, filename_prefix="metadynamics_convergence"):
    """Save all convergence data to pickle files"""
    
        
    # Extract directory from prefix and create if needed
    prefix_path = Path(filename_prefix)
    output_dir = prefix_path.parent
    base_name = prefix_path.name
    
    # Create directory if it doesn't exist
    if output_dir != Path('.'):  # Only create if not current directory
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {output_dir}")
        
    # Data structure to save
    save_data = {
        'metadata': {
            'n_walkers': len(walker_data),
            'walker_frame_counts': {walker_id: len(df) for walker_id, df in walker_data.items()},
            'state_labels': {
                1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
                5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
            },
            'parameters': {
                'window_size': 1000,  # Update with your actual window size
                'dt': 0.002,  # Update with your actual timestep
                'temperature': 2.5,  # kT value used in calculations
                'normalization': 'minimum'  # or whatever you used
            }
        },
        'raw_walker_data': walker_data,
        'individual_walker_convergence': convergence_results['individual'],
        'averaged_convergence': convergence_results['averaged']
    }
    
    # Save main data
    main_filename = f"{filename_prefix}.pkl"
    with open(main_filename, 'wb') as f:
        pickle.dump(save_data, f)
    
    print(f"Convergence data saved to: {main_filename}")
    
    
    return main_filename

# After your calculations, save the data
convergence_results = {
    'individual': walker_convergence,  # Individual walker convergence
    'averaged': averaged_convergence   # Averaged convergence
}

main_file = save_convergence_data(
    walker_data_colvar,  # or walker_data_output
    convergence_results,
    filename_prefix="convergence_data/8oppc_metadynamics_convergence"
)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_walker_data(walker_id, base_path='/Volumes/LaCie/STEPs/ConvergenceTest/walkers2'):
    """Load data from a single walker"""
    omegas_file = f'{base_path}/{walker_id}/output.txt'
    
    # Step 1: Read the column names from the line starting with "#! FIELDS"
    with open(omegas_file, 'r') as f:
        for line in f:
            if line.startswith('#! FIELDS'):
                columns = line.strip().split()[2:]  # Skip '#!', 'FIELDS'
                break
    
    # Step 2: Use `pd.read_csv` with `skiprows`
    def skip_every_20th(start, total_lines):
        return [i for i in range(start, total_lines) if i % 20 != 0]
    
    with open(omegas_file, 'r') as f:
        total_rows = sum(1 for line in f if not line.startswith('#'))
    
    df = pd.read_csv(
        omegas_file,
        delim_whitespace=True,
        comment='#',
        header=None,
        names=columns,
        skiprows=skip_every_20th(1, total_rows)
    )
    
    return df

def preprocess_walker_data(df):
    """Apply your preprocessing to a walker's data"""
    # Convert object columns to float
    object_columns = df.select_dtypes(include=['object']).columns
    df[object_columns] = df[object_columns].apply(pd.to_numeric, errors='coerce')
    
    # Convert to degrees
    df['om1'] = df['omega1'] * (180/np.pi)
    df['om2'] = df['omega2'] * (180/np.pi)
    df['om3'] = df['omega3'] * (180/np.pi)
    
    # Apply your conditions
    conditions = [
        ((df['om1'] >=-90) & (df['om1'] <=90)) & ((df['om2'] >=-90) & (df['om2'] <=90)) & ((df['om3'] >=-90) & (df['om3'] <=90)),
        ((df['om1'] >=-90) & (df['om1'] <=90)) & ((df['om2'] >=-90) & (df['om2'] <=90)) & ((df['om3'] <=-90) | (df['om3'] >=90)),
        ((df['om1'] >=-90) & (df['om1'] <=90)) & ((df['om2'] <=-90) | (df['om2'] >=90)) & ((df['om3'] >=-90) & (df['om3'] <=90)),
        ((df['om1'] <=-90) | (df['om1'] >=90)) & ((df['om2'] >=-90) & (df['om2'] <=90)) & ((df['om3'] >=-90) & (df['om3'] <=90)),
        ((df['om1'] <=-90) | (df['om1'] >=90)) & ((df['om2'] <=-90) | (df['om2'] >=90)) & ((df['om3'] >=-90) & (df['om3'] <=90)),
        ((df['om1'] <=-90) | (df['om1'] >=90)) & ((df['om2'] >=-90) & (df['om2'] <=90)) & ((df['om3'] <=-90) | (df['om3'] >=90)),
        ((df['om1'] >=-90) & (df['om1'] <=90)) & ((df['om2'] <=-90) | (df['om2'] >=90)) & ((df['om3'] <=-90) | (df['om3'] >=90)),
        ((df['om1'] <=-90) | (df['om1'] >=90)) & ((df['om2'] <=-90) | (df['om2'] >=90)) & ((df['om3'] <=-90) | (df['om3'] >=90))
    ]
    choices = [1,2,3,4,5,6,7,8]
    df['c'] = np.select(conditions, choices, default=0)
    
    return df

# Load and preprocess all walker data
print("Loading walker data...")
walker_data = {}
for walker_id in range(1, 9):
    print(f"Loading walker {walker_id}...")
    df_walker = load_walker_data(walker_id)
    df_walker = preprocess_walker_data(df_walker)
    walker_data[walker_id] = df_walker
    print(f"Walker {walker_id}: {len(df_walker)} frames")

# State labels
state_labels = {
    1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
    5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
}

# Option 1: Combine all walkers into one dataset
def combine_all_walkers(walker_data):
    """Combine all walker data into one large dataset"""
    combined_df = pd.concat([walker_data[i] for i in range(1, 9)], ignore_index=True)
    return combined_df

# Option 2: Calculate convergence for each walker separately, then average
def calculate_walker_convergence(walker_data, window_size=1000):
    """Calculate convergence for each walker separately"""
    walker_convergence = {}
    
    for walker_id in range(1, 9):
        print(f"Processing walker {walker_id}...")
        df = walker_data[walker_id]
        
        frames = range(window_size, len(df), window_size)
        convergence_data = {
            'frames': [],
            'populations': {state: [] for state in range(1, 9)},
            'free_energies': {state: [] for state in range(1, 9)}
        }
        
        for frame in frames:
            pops, fes = calculate_cumulative_populations(df, frame)
            convergence_data['frames'].append(frame)
            
            for state in range(1, 9):
                convergence_data['populations'][state].append(pops[state])
                convergence_data['free_energies'][state].append(fes[state])
        
        walker_convergence[walker_id] = convergence_data
    
    return walker_convergence

def calculate_cumulative_populations(df, up_to_frame):
    """Your existing function"""
    subset_df = df.iloc[:up_to_frame]
    global_min = np.exp(subset_df['pb.bias']/2.5).sum()
    
    populations = {}
    free_energies = {}
    
    for state in range(1, 9):
        state_data = subset_df[subset_df['c'] == state]
        if len(state_data) > 0:
            pop = np.exp(state_data['pb.bias']/2.5).sum() / global_min
            populations[state] = pop
            if pop > 0:
                free_energies[state] = -0.6 * np.log(pop)
            else:
                free_energies[state] = np.inf
        else:
            populations[state] = 0
            free_energies[state] = np.inf
    
    return populations, free_energies

# Calculate convergence for each walker
walker_convergence = calculate_walker_convergence(walker_data, window_size=1000)

# Calculate average convergence across walkers
def average_walker_convergence(walker_convergence):
    """Average convergence data across all walkers"""
    # Find common time points (minimum length across walkers)
    min_length = min(len(walker_convergence[i]['frames']) for i in range(1, 9))
    
    averaged_convergence = {
        'frames': walker_convergence[1]['frames'][:min_length],
        'populations': {state: [] for state in range(1, 9)},
        'free_energies': {state: [] for state in range(1, 9)},
        'pop_std': {state: [] for state in range(1, 9)},
        'fe_std': {state: [] for state in range(1, 9)}
    }
    
    for time_idx in range(min_length):
        for state in range(1, 9):
            # Collect values from all walkers at this time point
            pops = [walker_convergence[walker]['populations'][state][time_idx] 
                   for walker in range(1, 9)]
            fes = [walker_convergence[walker]['free_energies'][state][time_idx] 
                  for walker in range(1, 9)]
            
            # Remove infinite values for averaging
            finite_fes = [fe for fe in fes if fe != np.inf and not np.isnan(fe)]
            
            # Calculate averages and standard deviations
            averaged_convergence['populations'][state].append(np.mean(pops))
            averaged_convergence['pop_std'][state].append(np.std(pops))
            
            if finite_fes:
                averaged_convergence['free_energies'][state].append(np.mean(finite_fes))
                averaged_convergence['fe_std'][state].append(np.std(finite_fes))
            else:
                averaged_convergence['free_energies'][state].append(np.inf)
                averaged_convergence['fe_std'][state].append(0)
    
    return averaged_convergence

# Calculate averaged convergence
print("Averaging across walkers...")
averaged_convergence = average_walker_convergence(walker_convergence)

# Convert to time
dt = 0.002  # adjust according to your simulation
time_ps = [frame * dt for frame in averaged_convergence['frames']]

# Plot averaged convergence with walker-to-walker error bars
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

colors = plt.cm.tab10(np.linspace(0, 1, 8))

# Plot population convergence with walker variability
for i, state in enumerate(range(1, 9)):
    pops = averaged_convergence['populations'][state]
    errors = averaged_convergence['pop_std'][state]
    
    ax1.plot(time_ps, pops, label=f'{state_labels[state]}', color=colors[i], linewidth=2)
    ax1.fill_between(time_ps, 
                     [p - e for p, e in zip(pops, errors)],
                     [p + e for p, e in zip(pops, errors)],
                     color=colors[i], alpha=0.3)

ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Population')
ax1.set_title('Population Convergence (Averaged Across 8 Walkers)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Plot free energy convergence with walker variability
for i, state in enumerate(range(1, 9)):
    fes = [fe if fe != np.inf else np.nan for fe in averaged_convergence['free_energies'][state]]
    errors = averaged_convergence['fe_std'][state]
    
    ax2.plot(time_ps, fes, label=f'{state_labels[state]}', color=colors[i], linewidth=2)
    ax2.fill_between(time_ps,
                     [f - e if not np.isnan(f) else f for f, e in zip(fes, errors)],
                     [f + e if not np.isnan(f) else f for f, e in zip(fes, errors)],
                     color=colors[i], alpha=0.3)

ax2.set_xlabel('Time (ps)')
ax2.set_ylabel('Free Energy (kcal/mol)')
ax2.set_title('Free Energy Convergence (Averaged Across 8 Walkers)')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final averaged results
print("\nFinal averaged results across all walkers:")
for state in range(1, 9):
    final_pop = averaged_convergence['populations'][state][-1]
    final_pop_std = averaged_convergence['pop_std'][state][-1]
    final_fe = averaged_convergence['free_energies'][state][-1]
    final_fe_std = averaged_convergence['fe_std'][state][-1]
    
    print(f"{state_labels[state]}: P = {final_pop:.4f} ± {final_pop_std:.4f}, "
          f"dG = {final_fe:.2f} ± {final_fe_std:.2f} kcal/mol")

# Optional: Plot individual walker trajectories for comparison
def plot_individual_walkers(walker_convergence, state_to_plot=1):
    """Plot individual walker trajectories for a specific state"""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for walker_id in range(1, 9):
        convergence = walker_convergence[walker_id]
        time_ps_walker = [frame * dt for frame in convergence['frames']]
        pops = convergence['populations'][state_to_plot]
        
        ax.plot(time_ps_walker, pops, alpha=0.7, label=f'Walker {walker_id}')
    
    # Plot average
    avg_pops = averaged_convergence['populations'][state_to_plot]
    ax.plot(time_ps, avg_pops, 'k-', linewidth=3, label='Average')
    
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Population')
    ax.set_title(f'Individual Walker Convergence - {state_labels[state_to_plot]}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

# Example: Plot individual trajectories for CCC state
plot_individual_walkers(walker_convergence, state_to_plot=1)

main_file = save_convergence_data(
    walker_data,  # or walker_data_output
    convergence_results,
    filename_prefix="convergence_data/8omg_metadynamics_convergence"
)

In [ ]:
def load_individual_trajectory_data(trajectory_paths, trajectory_names=None):
    """
    Load data from individual long trajectories
    
    Parameters:
    trajectory_paths: list of file paths to trajectory files
    trajectory_names: optional list of names for each trajectory (e.g., ['Run1', 'Run2', ...])
    """
    
    if trajectory_names is None:
        trajectory_names = [f"Trajectory_{i+1}" for i in range(len(trajectory_paths))]
    
    trajectory_data = {}
    
    for i, (path, name) in enumerate(zip(trajectory_paths, trajectory_names)):
        print(f"Loading {name} from {path}...")
        
        try:
            # Use your existing loading logic
            with open(path, 'r') as f:
                for line in f:
                    if line.startswith('#! FIELDS'):
                        columns = line.strip().split()[2:]
                        break
            
            def skip_every_20th(start, total_lines):
                return [i for i in range(start, total_lines) if i % 20 != 0]
            
            with open(path, 'r') as f:
                total_rows = sum(1 for line in f if not line.startswith('#'))
            
            df = pd.read_csv(
                path,
                delim_whitespace=True,
                comment='#',
                header=None,
                names=columns,
                skiprows=skip_every_20th(1, total_rows)
            )
            
            # Preprocess the data
            df = preprocess_walker_data(df)  # Your existing preprocessing function
            trajectory_data[name] = df
            print(f"  {name}: {len(df)} frames loaded")
            
        except Exception as e:
            print(f"  ❌ Error loading {name}: {e}")
    
    return trajectory_data

In [ ]:
def calculate_individual_trajectory_convergence_with_blocks(trajectory_data, window_size=10000, n_blocks=5):
    """Calculate convergence for individual trajectories with block averaging errors"""
    trajectory_convergence = {}
    
    for traj_name, df in trajectory_data.items():
        print(f"Processing {traj_name}...")
        
        frames = range(window_size, len(df), window_size)
        convergence_data = {
            'frames': [],
            'populations': {state: [] for state in range(1, 9)},
            'free_energies': {state: [] for state in range(1, 9)},
            'pop_errors': {state: [] for state in range(1, 9)},
            'fe_errors': {state: [] for state in range(1, 9)}
        }
        
        for frame in frames:
            pops, fes, errors = calculate_cumulative_populations_with_block_error(df, frame, n_blocks)
            convergence_data['frames'].append(frame)
            
            for state in range(1, 9):
                convergence_data['populations'][state].append(pops[state])
                convergence_data['free_energies'][state].append(fes[state])
                convergence_data['pop_errors'][state].append(errors[state])
                
                # Error propagation for free energy
                if pops[state] > 0 and not np.isnan(errors[state]):
                    fe_error = (0.6 / pops[state]) * errors[state]
                    convergence_data['fe_errors'][state].append(fe_error)
                else:
                    convergence_data['fe_errors'][state].append(np.nan)
        
        trajectory_convergence[traj_name] = convergence_data
    
    return trajectory_convergence

def calculate_cumulative_populations_with_block_error(df, up_to_frame, n_blocks=5):
    """Calculate populations with block averaging error for a single trajectory"""
    subset_df = df.iloc[:up_to_frame]
    
    # Calculate overall populations and free energies
    global_min = np.exp(subset_df['pb.bias']/2.5).sum()
    
    populations = {}
    free_energies = {}
    
    for state in range(1, 9):
        state_data = subset_df[subset_df['c'] == state]
        if len(state_data) > 0:
            pop = np.exp(state_data['pb.bias']/2.5).sum() / global_min
            populations[state] = pop
            if pop > 0:
                free_energies[state] = -0.6 * np.log(pop)
            else:
                free_energies[state] = np.inf
        else:
            populations[state] = 0
            free_energies[state] = np.inf
    
    # Normalize free energies to minimum = 0
    finite_fes = {state: fe for state, fe in free_energies.items() 
                  if fe != np.inf and not np.isnan(fe)}
    if finite_fes:
        min_fe = min(finite_fes.values())
        for state in free_energies:
            if free_energies[state] != np.inf and not np.isnan(free_energies[state]):
                free_energies[state] -= min_fe
    
    # Calculate block averaging errors
    block_size = len(subset_df) // n_blocks
    
    if block_size < 100:  # Need minimum block size
        errors = {state: np.nan for state in range(1, 9)}
        return populations, free_energies, errors
    
    block_populations = {state: [] for state in range(1, 9)}
    
    for block in range(n_blocks):
        start_idx = block * block_size
        end_idx = (block + 1) * block_size if block < n_blocks - 1 else len(subset_df)
        block_df = subset_df.iloc[start_idx:end_idx]
        
        if len(block_df) == 0:
            continue
            
        global_min_block = np.exp(block_df['pb.bias']/2.5).sum()
        
        for state in range(1, 9):
            state_data = block_df[block_df['c'] == state]
            if len(state_data) > 0 and global_min_block > 0:
                pop = np.exp(state_data['pb.bias']/2.5).sum() / global_min_block
                block_populations[state].append(pop)
            else:
                block_populations[state].append(0)
    
    # Calculate standard error for each state
    errors = {}
    for state in range(1, 9):
        if len(block_populations[state]) > 1:
            errors[state] = np.std(block_populations[state]) / np.sqrt(len(block_populations[state]))
        else:
            errors[state] = np.nan
    
    return populations, free_energies, errors

In [ ]:
def plot_individual_trajectories_separate(trajectory_convergence, states_to_plot=None):
    """Plot each trajectory separately with its own block averaging errors"""
    
    state_labels = {
        1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
        5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
    }
    
    if states_to_plot is None:
        states_to_plot = range(1, 9)
    
    dt = 0.0004
    n_trajectories = len(trajectory_convergence)
    
    # Create subplots for each trajectory
    fig, axes = plt.subplots(n_trajectories, 2, figsize=(16, 5*n_trajectories))
    if n_trajectories == 1:
        axes = axes.reshape(1, -1)
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(states_to_plot)))
    
    for traj_idx, (traj_name, conv_data) in enumerate(trajectory_convergence.items()):
        time_ps = [frame * dt for frame in conv_data['frames']]
        
        # Population plot
        ax_pop = axes[traj_idx, 0]
        for i, state in enumerate(states_to_plot):
            pops = conv_data['populations'][state]
            errors = conv_data['pop_errors'][state]
            
            ax_pop.plot(time_ps, pops, label=f'{state_labels[state]}', 
                       color=colors[i], linewidth=2)
            ax_pop.fill_between(time_ps, 
                               [p - e if not np.isnan(e) else p for p, e in zip(pops, errors)],
                               [p + e if not np.isnan(e) else p for p, e in zip(pops, errors)],
                               color=colors[i], alpha=0.3)
        
        ax_pop.set_xlabel('Time (ns)')
        ax_pop.set_ylabel('Population')
        ax_pop.set_title(f'{traj_name} - Population Convergence')
        ax_pop.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax_pop.grid(True, alpha=0.3)
        ax_pop.set_ylim(0, 1)
        
        # Free energy plot
        ax_fe = axes[traj_idx, 1]
        for i, state in enumerate(states_to_plot):
            fes = [fe if fe != np.inf else np.nan for fe in conv_data['free_energies'][state]]
            errors = conv_data['fe_errors'][state]
            
            ax_fe.plot(time_ps, fes, label=f'{state_labels[state]}', 
                      color=colors[i], linewidth=2)
            ax_fe.fill_between(time_ps,
                              [f - e if not np.isnan(e) and not np.isnan(f) else f 
                               for f, e in zip(fes, errors)],
                              [f + e if not np.isnan(e) and not np.isnan(f) else f 
                               for f, e in zip(fes, errors)],
                              color=colors[i], alpha=0.3)
        
        ax_fe.set_xlabel('Time (ns)')
        ax_fe.set_ylabel('Relative Free Energy (kcal/mol)')
        ax_fe.set_title(f'{traj_name} - Free Energy Convergence')
        ax_fe.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax_fe.grid(True, alpha=0.3)
        ax_fe.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

def plot_trajectory_comparison_overlay(trajectory_convergence, states_to_plot=None):
    """Plot all trajectories overlaid for comparison (without averaging)"""
    
    state_labels = {
        1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
        5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
    }
    
    if states_to_plot is None:
        states_to_plot = range(1, 9)
    
    dt = 0.0004
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(states_to_plot)))
    line_styles = ['-', '--', '-.', ':']
    
    # Population comparison
    for traj_idx, (traj_name, conv_data) in enumerate(trajectory_convergence.items()):
        time_ps = [frame * dt for frame in conv_data['frames']]
        line_style = line_styles[traj_idx % len(line_styles)]
        
        for i, state in enumerate(states_to_plot):
            pops = conv_data['populations'][state]
            label = f'{state_labels[state]} ({traj_name})' if len(trajectory_convergence) <= 2 else None
            if traj_idx == 0:
                label = f'{state_labels[state]}'
            
            ax1.plot(time_ps, pops, color=colors[i], linestyle=line_style,
                    linewidth=2, alpha=0.8, label=label)
    
    ax1.set_xlabel('Time (ns)')
    ax1.set_ylabel('Population')
    ax1.set_title('Population Convergence - All Trajectories')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # Free energy comparison
    for traj_idx, (traj_name, conv_data) in enumerate(trajectory_convergence.items()):
        time_ps = [frame * dt for frame in conv_data['frames']]
        line_style = line_styles[traj_idx % len(line_styles)]
        
        for i, state in enumerate(states_to_plot):
            fes = [fe if fe != np.inf else np.nan for fe in conv_data['free_energies'][state]]
            label = f'{state_labels[state]} ({traj_name})' if len(trajectory_convergence) <= 2 else None
            if traj_idx == 0:
                label = f'{state_labels[state]}'
            
            ax2.plot(time_ps, fes, color=colors[i], linestyle=line_style,
                    linewidth=2, alpha=0.8, label=label)
    
    ax2.set_xlabel('Time (ns)')
    ax2.set_ylabel('Relative Free Energy (kcal/mol)')
    ax2.set_title('Free Energy Convergence - All Trajectories')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def print_individual_trajectory_results(trajectory_convergence):
    """Print final results for each trajectory separately"""
    
    state_labels = {
        1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
        5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
    }
    
    for traj_name, conv_data in trajectory_convergence.items():
        print(f"\n{'='*60}")
        print(f"Final Results for {traj_name}")
        print(f"{'='*60}")
        
        for state in range(1, 9):
            final_pop = conv_data['populations'][state][-1]
            final_pop_err = conv_data['pop_errors'][state][-1]
            final_fe = conv_data['free_energies'][state][-1]
            final_fe_err = conv_data['fe_errors'][state][-1]
            
            if final_fe != np.inf:
                print(f"{state_labels[state]}: P = {final_pop:.4f} ± {final_pop_err:.4f}, "
                      f"ΔG = {final_fe:.2f} ± {final_fe_err:.2f} kcal/mol")
            else:
                print(f"{state_labels[state]}: P = {final_pop:.4f} ± {final_pop_err:.4f}, "
                      f"ΔG = ∞ kcal/mol (not observed)")
        
        # Find reference state for this trajectory
        finite_states = {s: conv_data['free_energies'][s][-1] 
                        for s in range(1, 9) 
                        if conv_data['free_energies'][s][-1] != np.inf}
        
        if finite_states:
            ref_state = min(finite_states, key=finite_states.get)
            print(f"Reference state (ΔG = 0): {state_labels[ref_state]}")

In [ ]:
# 1. Load individual trajectories
trajectory_paths = [
    "/Volumes/LaCie/STEPs/ConvergenceTest/omega/data/COLVAR",
    "/Volumes/LaCie/STEPs/ConvergenceTest/ophi/data/COLVAR", 
    "/Volumes/LaCie/STEPs/ConvergenceTest/ophps/data/COLVAR",
    "/Volumes/LaCie/STEPs/ConvergenceTest/oppc/data/COLVAR"
]

trajectory_names = ["omega", "ophi", "opp", "oppc"]

individual_trajectories = load_individual_trajectory_data(trajectory_paths, trajectory_names)

# 2. Calculate convergence with block averaging for each trajectory
individual_traj_convergence = calculate_individual_trajectory_convergence_with_blocks(
    individual_trajectories, 
    window_size=10000,  # Adjust based on trajectory length
    n_blocks=5
)

# 3. Plot each trajectory separately
plot_individual_trajectories_separate(individual_traj_convergence)

# 4. Plot all trajectories overlaid for comparison
plot_trajectory_comparison_overlay(individual_traj_convergence)

# 5. Print convergence results
print_individual_trajectory_results(individual_traj_convergence)

In [ ]:
def save_individual_trajectories_separate(trajectory_data, trajectory_convergence, filename_prefixes):
    """
    Save each trajectory as a separate pickle file with custom prefixes
    
    Parameters:
    trajectory_data: dict of trajectory dataframes
    trajectory_convergence: dict of convergence results
    filename_prefixes: dict mapping trajectory names to their desired filename prefixes
                      e.g., {"8OPPC_Long_Run1": "convergence_data/omega", 
                             "8OPPC_Long_Run2": "convergence_data/ophi", ...}
    """
    
    saved_files = {}
    
    for traj_name in trajectory_data.keys():
        print(f"Saving {traj_name}...")
        
        # Get the prefix for this trajectory
        if traj_name not in filename_prefixes:
            print(f"Warning: No prefix specified for {traj_name}, skipping...")
            continue
            
        filename_prefix = filename_prefixes[traj_name]
        
        # Extract directory from prefix and create if needed
        prefix_path = Path(filename_prefix)
        output_dir = prefix_path.parent
        base_name = prefix_path.name
        
        if output_dir != Path('.'):
            output_dir.mkdir(parents=True, exist_ok=True)
            print(f"Created directory: {output_dir}")
        
        # Create data structure for this trajectory
        traj_save_data = {
            'metadata': {
                'trajectory_name': traj_name,
                'frame_count': len(trajectory_data[traj_name]),
                'state_labels': {
                    1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
                    5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
                },
                'parameters': {
                    'window_size': 10000,  # Update with actual value
                    'n_blocks': 5,        # Update with actual value
                    'dt': 0.0004,
                    'temperature': 2.5,
                    'normalization': 'minimum'
                }
            },
            'raw_data': trajectory_data[traj_name],
            'convergence_data': trajectory_convergence[traj_name]
        }
        
        # Create filename (no timestamp)
        main_filename = output_dir / f"{base_name}.pkl"
        
        # Save file
        with open(main_filename, 'wb') as f:
            pickle.dump(traj_save_data, f)
        
        saved_files[traj_name] = str(main_filename)
        print(f"  Saved: {main_filename}")
    
    # Also save a summary file
    summary_data = {
        'metadata': {
            'n_trajectories': len(trajectory_data),
            'trajectory_names': list(trajectory_data.keys()),
            'trajectory_files': saved_files,
            'state_labels': {
                1: "CCC", 2: "CCT", 3: "CTC", 4: "TCC",
                5: "TTC", 6: "TCT", 7: "CTT", 8: "TTT"
            }
        },
        'summary_results': {}
    }
    
    # Add final results summary
    for traj_name, conv_data in trajectory_convergence.items():
        if traj_name in saved_files:  # Only include saved trajectories
            summary_data['summary_results'][traj_name] = {
                'final_populations': {state: conv_data['populations'][state][-1] for state in range(1, 9)},
                'final_free_energies': {state: conv_data['free_energies'][state][-1] for state in range(1, 9)},
                'final_pop_errors': {state: conv_data['pop_errors'][state][-1] for state in range(1, 9)},
                'final_fe_errors': {state: conv_data['fe_errors'][state][-1] for state in range(1, 9)}
            }
    
    # Save summary in the same directory as the first trajectory
    first_prefix = list(filename_prefixes.values())[0]
    summary_dir = Path(first_prefix).parent
    summary_filename = summary_dir / "trajectory_summary.pkl"
    
    with open(summary_filename, 'wb') as f:
        pickle.dump(summary_data, f)
    
    print(f"\nSummary file saved: {summary_filename}")
    print(f"Total files saved: {len(saved_files) + 1}")
    
    return saved_files, str(summary_filename)

# Usage with your specific prefixes:
filename_prefixes = {
    "omega": "/Volumes/LaCie/STEPs/ConvergenceTest/convergence_data/omega",
    "ophi": "/Volumes/LaCie/STEPs/ConvergenceTest/convergence_data/ophi", 
    "opp": "/Volumes/LaCie/STEPs/ConvergenceTest/convergence_data/opp",
    "oppc": "/Volumes/LaCie/STEPs/ConvergenceTest/convergence_data/oppc"
}

saved_files, summary_file = save_individual_trajectories_separate(
    individual_trajectories,
    individual_traj_convergence,
    filename_prefixes
)